# Analyse exploratoire des données

In [17]:
import json
import xml.etree.ElementTree as ET
from collections import Counter
import numpy as np

## Nombre d'occurrences des noeuds

In [18]:
# Charger le dataset complet
with open('bt_dataset.json', 'r') as f:
    dataset = json.load(f)

vocabulaire_noeuds = Counter()
longueurs_prompt_mots = []
longueurs_xml_mots = []
parse_errors = 0
exemples_parse_errors = []

for idx, item in enumerate(dataset):
    # 1. Analyse des longueurs (approximation en mots)
    prompt = item.get('input', '')
    xml_text = item.get('output', '')
    longueurs_prompt_mots.append(len(prompt.split()))
    longueurs_xml_mots.append(len(xml_text.split()))
    
    # 2. Extraction du vocabulaire XML
    try:
        # Si la sortie contient déjà une déclaration XML, on ne peut pas l'encapsuler directement.
        xml_text_sans_decl = xml_text.replace('<?xml version="1.0"?>', '').strip()
        root = ET.fromstring(f"<dummy_root>{xml_text_sans_decl}</dummy_root>")
        for elem in root.iter():
            if elem.tag not in ['dummy_root', 'root', 'BehaviorTree']:
                vocabulaire_noeuds[elem.tag] += 1
    except ET.ParseError as e:
        parse_errors += 1
        if len(exemples_parse_errors) < 5:
            exemples_parse_errors.append((idx, str(e)))

print("___ Statistiques du Dataset BTGenBot ___")
print(f"Nombre d'exemples : {len(dataset)}")
print(f"Longueur moyenne Input (mots) : {np.mean(longueurs_prompt_mots):.0f}")
print(f"Longueur moyenne XML (mots) : {np.mean(longueurs_xml_mots):.0f}")
print(f"Exemples XML non parseables : {parse_errors}")
if exemples_parse_errors:
    print("Exemples d'erreurs XML :")
    for idx, err in exemples_parse_errors:
        print(f"- index {idx}: {err}")

print(f"\nVocabulaire des Nœuds (Top 15) :")
for noeud, freq in vocabulaire_noeuds.most_common(15):
    print(f"- {noeud} : {freq} occurrences")

___ Statistiques du Dataset BTGenBot ___
Nombre d'exemples : 594
Longueur moyenne Input (mots) : 115
Longueur moyenne XML (mots) : 141
Exemples XML non parseables : 23
Exemples d'erreurs XML :
- index 5: not well-formed (invalid token): line 2, column 10
- index 6: not well-formed (invalid token): line 2, column 10
- index 47: not well-formed (invalid token): line 2, column 10
- index 51: XML or text declaration not at start of entity: line 1, column 12
- index 61: XML or text declaration not at start of entity: line 1, column 12

Vocabulaire des Nœuds (Top 15) :
- Action : 5458 occurrences
- input_port : 3133 occurrences
- Condition : 1817 occurrences
- Sequence : 1444 occurrences
- output_port : 897 occurrences
- SubTree : 763 occurrences
- Fallback : 645 occurrences
- SetBlackboard : 361 occurrences
- SequenceStar : 327 occurrences
- inout_port : 308 occurrences
- TreeNodesModel : 225 occurrences
- ClearEntireCostmap : 174 occurrences
- ForceSuccess : 162 occurrences
- TextToSpeechA

### Petit nettoyage 23 fichiers ne sont pas valides

In [19]:
# Filtrer les exemples avec XML invalide et sauvegarder un dataset nettoyé
def xml_valide_pour_eda(xml_text: str) -> bool:
    try:
        xml_text_sans_decl = xml_text.replace('<?xml version="1.0"?>', '').strip()
        ET.fromstring(f"<dummy_root>{xml_text_sans_decl}</dummy_root>")
        return True
    except ET.ParseError:
        return False

dataset_filtre = [
    item for item in dataset
    if xml_valide_pour_eda(item.get('output', ''))
]

indices_supprimes = [
    idx for idx, item in enumerate(dataset)
    if not xml_valide_pour_eda(item.get('output', ''))
]

with open('bt_dataset_filtered.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_filtre, f, ensure_ascii=False, indent=2)

print('___ Filtrage XML du dataset ___')
print(f"Taille originale : {len(dataset)}")
print(f"Taille filtrée : {len(dataset_filtre)}")
print(f"Exemples retirés : {len(indices_supprimes)}")
print("Fichier écrit : bt_dataset_filtered.json")
print("Aperçu indices retirés (10 max) :", indices_supprimes[:10])

___ Filtrage XML du dataset ___
Taille originale : 594
Taille filtrée : 571
Exemples retirés : 23
Fichier écrit : bt_dataset_filtered.json
Aperçu indices retirés (10 max) : [5, 6, 47, 51, 61, 89, 95, 98, 118, 211]


## Noeuds les plus communs (présents dans beaucoup de XML différents)

In [20]:
# Nœuds présents au moins une fois dans tous les XML valides
dataset_source = dataset_filtre if 'dataset_filtre' in globals() else [
    item for item in dataset if xml_valide_pour_eda(item.get('output', ''))
]

from collections import Counter
doc_freq = Counter()
noeuds_communs = None

for item in dataset_source:
    xml_text = item.get('output', '')
    xml_text_sans_decl = xml_text.replace('<?xml version="1.0"?>', '').strip()
    root = ET.fromstring(f"<dummy_root>{xml_text_sans_decl}</dummy_root>")

    # Ensemble des tags présents dans cet XML (présence/absence, pas le nombre d'occurrences)
    tags_xml = {
        elem.tag for elem in root.iter()
        if elem.tag not in ['dummy_root', 'root']
    }

    for tag in tags_xml:
        doc_freq[tag] += 1

    if noeuds_communs is None:
        noeuds_communs = set(tags_xml)
    else:
        noeuds_communs &= tags_xml

noeuds_communs = sorted(noeuds_communs) if noeuds_communs else []
n_xml = len(dataset_source)

print('___ Nœuds présents dans 100% des XML valides ___')
print(f"Nombre de XML considérés : {n_xml}")
print(f"Nombre de nœuds communs : {len(noeuds_communs)}")
for tag in noeuds_communs:
    print(f"- {tag}")

print('\n___ Couverture par fichier (Top 15) ___')
for tag, c in doc_freq.most_common(15):
    print(f"- {tag}: {c}/{n_xml} ({100*c/n_xml:.2f}%)")

___ Nœuds présents dans 100% des XML valides ___
Nombre de XML considérés : 571
Nombre de nœuds communs : 0

___ Couverture par fichier (Top 15) ___
- BehaviorTree: 570/571 (99.82%)
- Sequence: 438/571 (76.71%)
- Action: 240/571 (42.03%)
- TreeNodesModel: 225/571 (39.40%)
- Fallback: 188/571 (32.92%)
- Condition: 145/571 (25.39%)
- input_port: 137/571 (23.99%)
- FollowPath: 132/571 (23.12%)
- ComputePathToPose: 128/571 (22.42%)
- SubTree: 124/571 (21.72%)
- Repeat: 119/571 (20.84%)
- PipelineSequence: 106/571 (18.56%)
- output_port: 106/571 (18.56%)
- SequenceStar: 105/571 (18.39%)
- RateController: 99/571 (17.34%)


### Extraction des "Skills" Métier

Pour extraire cette liste de skills à partir du dataset, la méthode la plus simple consiste à filtrer le vocabulaire_noeuds (ou doc_freq) en excluant les tags correspondant aux nœuds de contrôle standards de BehaviorTree.CPP. Ce qui restera sera, par déduction, les skills métiers.

In [21]:
# %%
import xml.etree.ElementTree as ET
from collections import Counter

# 1. Enrichissement de la liste des nœuds standards, décorateurs et structurels
btcpp_standard_nodes = {
    # --- Nœuds de contrôle et décorateurs standards ---
    'BehaviorTree', 'Sequence', 'SequenceStar', 'Fallback', 'FallbackStar', 
    'ReactiveSequence', 'ReactiveFallback', 'Parallel', 'PipelineSequence', 
    'RecoveryNode', 'RoundRobin', 'Inverter', 'RetryUntilSuccessful', 
    'RetryUntilSuccesful', # Inclusion de la faute de frappe fréquente dans ROS2
    'KeepRunningUntilFailure', 'ForceSuccess', 'ForceFailure', 'AlwaysSuccess', 
    'AlwaysFailure', 'Timeout', 'Delay', 'RateController', 'DistanceController', 
    'SpeedController', 'SubTree', 'SubTreePlus', 'Repeat', 'IfThenElse', 'WhileDo',
    
    # --- Manipulation du Blackboard (built-in BTCPP) ---
    'SetBlackboard', 'BlackboardCheckInt', 'BlackboardCheckDouble', 
    'BlackboardCheckString', 'BlackboardCheckBool', 'BB_Precondition',
    
    # --- Structure XML (Bruit) ---
    'root', 'dummy_root', 'TreeNodesModel', 'Action', 'Condition', 
    'Control', 'Decorator', 'input_port', 'output_port', 'inout_port'
}

# On filtre pour ne garder que les nœuds métiers
skills_potentiels = Counter()

for item in dataset_source:
    xml_text = item.get('output', '')
    try:
        xml_text_sans_decl = xml_text.replace('<?xml version="1.0"?>', '').strip()
        root = ET.fromstring(f"<dummy_root>{xml_text_sans_decl}</dummy_root>")
        
        # 2. Astuce clé : Supprimer complètement le bloc TreeNodesModel de l'arbre
        # Cela évite de compter les déclarations de skills et de ports comme des exécutions
        for model in root.findall('.//TreeNodesModel'):
            # On retire ce sous-arbre du parent (dummy_root ou root)
            # Pour gérer les XML fragmentés, on cherche le parent
            parent = root.find(f".//TreeNodesModel/..")
            if parent is not None:
                parent.remove(model)
        
        # 3. Comptage sur l'arbre nettoyé
        for elem in root.iter():
            if elem.tag not in btcpp_standard_nodes:
                skills_potentiels[elem.tag] += 1
                
    except ET.ParseError:
        continue

print('\n___ Liste des Skills Potentiels (Métier) nettoyée ___')
print(f"Nombre total de skills uniques identifiés : {len(skills_potentiels)}")
print("Top 25 des skills les plus fréquents :")
for skill, freq in skills_potentiels.most_common(25):
    print(f"- {skill} : {freq} occurrences")


___ Liste des Skills Potentiels (Métier) nettoyée ___
Nombre total de skills uniques identifiés : 314
Top 25 des skills les plus fréquents :
- ClearEntireCostmap : 174 occurrences
- TextToSpeechActionClient : 156 occurrences
- ComputePathToPose : 143 occurrences
- FollowPath : 136 occurrences
- GoalUpdated : 83 occurrences
- Wait : 76 occurrences
- MoveAhead : 72 occurrences
- TrackAction : 64 occurrences
- Spin : 61 occurrences
- Turn : 41 occurrences
- MoveBase : 37 occurrences
- AntennaAction : 33 occurrences
- Nav2Client : 30 occurrences
- PlanManipulatorPath : 29 occurrences
- MoveManipulator : 29 occurrences
- RobotSpin : 29 occurrences
- TextCompareAction : 27 occurrences
- NavigateToWp : 25 occurrences
- BaseMovement : 24 occurrences
- HumanPoseDetect : 23 occurrences
- SmileAction : 21 occurrences
- GoalUpdater : 21 occurrences
- CheckBlackboard : 20 occurrences
- Print : 20 occurrences
- ChangeParam : 20 occurrences


## Classification des BT selon leur skills

In [22]:
def categoriser_bt_ameliore(xml_text: str) -> str:
    """Catégorise le BT en analysant la présence de tags métiers spécifiques."""
    try:
        xml_text_sans_decl = xml_text.replace('<?xml version="1.0"?>', '').strip()
        root = ET.fromstring(f"<dummy_root>{xml_text_sans_decl}</dummy_root>")
        tags = {elem.tag for elem in root.iter()}
    except:
        return "Erreur Parsing"

    # 1. Manipulation et Bras Robotique (Tâches 4, 6, 7, 9)
    manipulation_skills = {'PlanManipulatorPath', 'MoveManipulator'}
    if tags.intersection(manipulation_skills):
        # S'il y a aussi de la nav, c'est une mission hybride (ex: Tâche 4)
        if tags.intersection({'ComputePathToPose', 'NavigateToWp', 'MoveBase'}):
            return "Navigation + Manipulation"
        return "Manipulation / Bras Robotique"

    # 2. Vision Active et Interaction (Tâche 7 et suppléments)
    vision_interaction_skills = {'HumanPoseDetect', 'SmileAction', 'TextToSpeechActionClient'}
    if tags.intersection(vision_interaction_skills):
        return "Vision Active / Interaction Humain-Robot"

    # 3. Navigation (Tâches 1, 2, 3, 5)
    nav_skills = {'ComputePathToPose', 'FollowPath', 'MoveBase', 'MoveAhead', 'NavigateToWp', 'Nav2Client', 'BaseMovement'}
    if tags.intersection(nav_skills):
        recovery_skills = {'ClearEntireCostmap', 'Spin', 'Wait', 'Turn', 'RobotSpin'}
        if tags.intersection(recovery_skills):
            return "Navigation avec Recovery/Fallback"
        return "Navigation Standard"
        
    # 4. Tracking et cas particuliers
    if 'TrackAction' in tags or 'AntennaAction' in tags:
        return "Tracking / Action Spécifique"
        
    return "Autre / Non classifié"

# Application sur le dataset propre
distribution_taches = Counter()
for item in dataset_source: # Assurez-vous d'utiliser le dataset filtré des erreurs de parsing
    cat = categoriser_bt_ameliore(item.get('output', ''))
    distribution_taches[cat] += 1

print('\n___ Distribution des Tâches (Heuristique affinée) ___')
for cat, freq in distribution_taches.most_common():
    print(f"- {cat} : {freq} exemples")


___ Distribution des Tâches (Heuristique affinée) ___
- Autre / Non classifié : 352 exemples
- Navigation Standard : 118 exemples
- Navigation avec Recovery/Fallback : 67 exemples
- Vision Active / Interaction Humain-Robot : 30 exemples
- Manipulation / Bras Robotique : 2 exemples
- Tracking / Action Spécifique : 2 exemples


## Conclusion

- Dominance de la navigation spatiale : La majorité des arbres gèrent des déplacements (Nav2, Waypoints, Recovery). Ce qui est positif pour notre projet comme nous l'avons déjà précisé  car NAV4RAIL est une stack de navigation autonome pour robots ferroviaires. Le modèle va donc s'entraîner sur des concepts logiques très proches de notre cible.
- Un domaine d'exécution ouvert : Le dataset est extrêmement hétérogène et exploite un vocabulaire ouvert de 314 skills uniques (incluant de la manipulation, de la vision, du contrôle système...).
- Validation de la stratégie du "1er Entonnoir" : Si un LLM compact de 7B paramètres parvient à apprendre la syntaxe BTCPP au milieu de ces centaines de skills , alors le restreindre par la suite au domaine contraint de la SNCF (un catalogue fermé de 28 skills typés ) sera une tâche beaucoup plus facile nécessitant très peu d'exemples (cf présentation BTGenBot)

# Conversion JSONL et Adaptation du Prompt pour le Fine-Tuning

Pourquoi faire ?

L'article BTGenBot (2024) utilise le format Alpaca basique (instruction, input, output). Cependant, pour notre fine-tunings, nous cherchons à utiliser des modèles plus récents basé sur le Chat (LLaMA-3.1-8B, Qwen2.5-7B). Ces modèles nécessitent un Chat Template strict avec des rôles (system, user, assistant).

In [24]:
import json

input_file = 'bt_dataset_filtered.json'
output_file = 'bt_dataset_chat.jsonl'

with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
    dataset = json.load(f_in)
    
    for item in dataset:
        # 1. Adaptation : L'instruction générique devient le System Prompt.
        # Note pour la Phase 2 : C'est ici que la SNCF pourra injecter son catalogue YAML de 28 skills
        system_prompt = item.get("instruction", "You will be provided a summary of a task performed by a robot, and your objective is to express this task as a behavior tree in XML format.")
        
        user_prompt = item.get("input", "")
        assistant_response = item.get("output", "")
        
        # 2. Formatage conversationnel OpenAI / Hugging Face (TRL)
        chat_format = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response}
            ]
        }
        
        # 3. Écriture en JSONL (une ligne par exemple)
        f_out.write(json.dumps(chat_format, ensure_ascii=False) + '\n')

print(f"___ Préparation terminée ___")
print(f"Fichier prêt pour le fine-tuning SFT : {output_file}")

___ Préparation terminée ___
Fichier prêt pour le fine-tuning SFT : bt_dataset_chat.jsonl
